In [1]:
from datetime import datetime
from typing import Optional
import pybamm
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from netlist_utils import setup_circuit, make_lcapy_circuit
from battery import LiionBatteryPack, PybammBattery, CLCBattery
import vessim as vs

import nest_asyncio
nest_asyncio.apply()

In [53]:
class ExperimentController(vs.Controller):
    def __init__(self):
        super().__init__()

    def step(self, time: datetime, p_delta: float, e_delta: float, state: dict) -> None:
        self.set_parameters["policy:min_soc"] = 0.6 if (time.hour > 11) or time.hour < 1  else 0.3
        self.set_parameters["policy:charge_power"] = 3200 if (time.day == 12 and time.hour == 11) else 0

In [42]:
class MinSocPolicy(vs.MicrogridPolicy):
    def __init__(self, charge_power: Optional[float] = None, min_soc = 0):
        self.charge_power = charge_power if charge_power else 0.0
        self.min_soc = min_soc

    def apply(self, p_delta: float, duration: int, storage: Optional[vs.Storage] = None) -> float:
        assert storage is not None
        energy_delta = p_delta * duration
        if self.charge_power:
            energy_delta -= storage.update(self.charge_power, duration)
        elif self.min_soc >= storage.soc() and p_delta <= 0:
            storage.update(0.0, duration)
        else:
            energy_delta -= storage.update(p_delta, duration)

        return energy_delta

    def state(self) -> dict:
        return {
            "charge_power": self.charge_power,
            "min_soc": self.min_soc
        }


In [36]:
def default_battery_models(initial_soc, np, ns, step_size):
    simple_battery = vs.SimpleBattery(capacity=18.87 * np * ns, initial_soc=initial_soc)
    clc_battery = CLCBattery(number_of_cells=ns*np, initial_soc=initial_soc)
    pybamm_battery = PybammBattery(
        model_type=pybamm.lithium_ion.SPMe,
        initial_soc=initial_soc,
        number_of_cells=ns*np,
        parameter_values=pybamm.ParameterValues("Chen2020"),
        output_variables=["Current [A]", "Voltage [V]", "Power [W]", "Discharge energy [W.h]"],
    )
    liion_battery_pack = LiionBatteryPack(
        model_type=pybamm.lithium_ion.SPMe,
        netlist=setup_circuit(Np=np, Ns=ns, I=0.0),
        step_size=step_size,
        initial_soc=initial_soc,
        parameter_values=pybamm.ParameterValues("Chen2020"),
    )
    return [simple_battery, clc_battery, pybamm_battery, liion_battery_pack]

In [37]:

def degredation_battery_models(initial_soc, np, ns, step_size):
    simple_battery = vs.SimpleBattery(capacity=18.87 * np * ns, initial_soc=initial_soc)
    clc_battery = CLCBattery(number_of_cells=ns*np, initial_soc=initial_soc)
    pybamm_battery = PybammBattery(
        model_type=pybamm.lithium_ion.SPMe,
        options={
            "SEI": "solvent-diffusion limited",
            "SEI porosity change": "true",
            "lithium plating": "partially reversible",
            "lithium plating porosity change": "true",  # alias for "SEI porosity change"
            "particle mechanics": ("swelling and cracking", "swelling only"),
            "SEI on cracks": "true",
            "loss of active material": "stress-driven",
        },
        initial_soc=initial_soc,
        number_of_cells=ns*np,
        parameter_values=pybamm.ParameterValues("OKane2022"),
        output_variables=["Voltage [V]", "Current [A]", "Power [W]"]
    )
    liion_battery_pack = LiionBatteryPack(
        model_type=pybamm.lithium_ion.SPMe,
        options={
            "SEI": "solvent-diffusion limited",
            "SEI porosity change": "true",
            "lithium plating": "partially reversible",
            "lithium plating porosity change": "true",  # alias for "SEI porosity change"
            "particle mechanics": ("swelling and cracking", "swelling only"),
            "SEI on cracks": "true",
            "loss of active material": "stress-driven",
        },
        netlist=setup_circuit(Np=np, Ns=ns, I=0.0),
        step_size=step_size,
        initial_soc=initial_soc,
        parameter_values=pybamm.ParameterValues("OKane2022"),
    )
    return [simple_battery, clc_battery, pybamm_battery, liion_battery_pack]

In [6]:
liion_battery = default_battery_models(step_size=10, np=4, ns=4, initial_soc=1)[3]
for i in range(10):
    liion_battery.update(-18.78 * 16, 10)
circuit = make_lcapy_circuit(liion_battery.netlist)
circuit.draw("../thesis/Images/pack_circuit.pdf", cpt_size=1, dpi=150, node_spacing=1.5, style="american")

In [26]:
STEP_SIZE = 10
C_RATES_DISCHARGE = [0.1, 0.2, 0.5, 1.0, 1.5, 2.0]
C_RATES_CHARGE = [0.1, 0.2, 0.3, 0.5, 0.7, 1.0]

In [65]:
for c_rate in C_RATES_DISCHARGE:
    power = -18.87 * c_rate * 256
    environment = vs.Environment(sim_start="2022-07-03 09:00:00")
    for battery in default_battery_models(initial_soc=1, np=16, ns=16, step_size=STEP_SIZE)[:4]:
        monitor = vs.Monitor(outfile=f"results/{type(battery).__name__}_Discharge{c_rate}C.csv")
        environment.add_microgrid(
            actors=[vs.Actor(name=f"MockActor:{type(battery).__name__}", signal=vs.MockSignal(value=power))],
            storage=battery,
            controllers=[monitor],
            step_size=STEP_SIZE,
        )
    if c_rate < 1.5:
        environment.run(until=(1 / c_rate) * 3800)
    else:
        environment.run(until=2 / 3 * 3800)

2024-07-16 16:32:26.825 | INFO     | mosaik.scenario:start:311 - Starting "Actor" as "MockActor:SimpleBattery" ...
2024-07-16 16:32:26.826 | INFO     | mosaik.scenario:start:311 - Starting "Grid" as "Grid-0" ...
2024-07-16 16:32:26.826 | INFO     | mosaik.scenario:start:311 - Starting "Controller" as "Monitor-20" ...
2024-07-16 16:32:26.827 | INFO     | mosaik.scenario:start:311 - Starting "Storage" as "Storage-0" ...
2024-07-16 16:32:26.827 | INFO     | mosaik.scenario:start:311 - Starting "Actor" as "MockActor:CLCBattery" ...
2024-07-16 16:32:26.828 | INFO     | mosaik.scenario:start:311 - Starting "Grid" as "Grid-1" ...
2024-07-16 16:32:26.828 | INFO     | mosaik.scenario:start:311 - Starting "Controller" as "Monitor-21" ...
2024-07-16 16:32:26.828 | INFO     | mosaik.scenario:start:311 - Starting "Storage" as "Storage-1" ...
2024-07-16 16:32:26.829 | INFO     | mosaik.scenario:start:311 - Starting "Actor" as "MockActor:PybammBattery" ...
2024-07-16 16:32:26.830 | INFO     | mosaik.

In [27]:
fig = make_subplots(rows=3, cols=2, shared_yaxes=True, vertical_spacing=0.07, horizontal_spacing=0.05, subplot_titles=tuple(f"C-Rate {i}" for i in C_RATES_DISCHARGE))
fig.update_layout(
    width=800,
    height=800,
    margin={"r": 0, "l": 0, "t": 20, "b": 0},
    paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
    font_size=14,
    legend={"xanchor": "right", "x": 1},
)
fig.update_xaxes(
    {"showline": True, "linewidth": 1, "linecolor": "black", "ticks":"outside"}
)
fig.update_yaxes(
    {"showline": True, "linewidth": 1, "linecolor": "black", "ticks":"outside"}
)
showlegend = True
batteries = ["SimpleBattery", "CLCBattery", "PybammBattery", "LiionBatteryPack"]
colors = ["Blue", "Red", "Green", "Purple"]
for i, c_rate in enumerate(C_RATES_DISCHARGE):
    col = i % 2 + 1
    row = i // 2 + 1
    for battery, color in zip(batteries, colors[:4]):
        df = pd.read_csv(f"results/{battery}_Discharge{c_rate}C.csv")
        fig.add_trace(
            go.Scatter(x=[i * STEP_SIZE / 3600 for i in range(len(df.index))], y=df["storage.soc"] * 100, showlegend=showlegend, line_color=color, name=battery), row=row, col=col
        )
    fig.update_xaxes(row=row, col=col, title="", ticksuffix="h")
    showlegend = False
for i in range(3):
    fig.update_yaxes(title="$\\text{State-of-Charge}$", ticksuffix = "%", row=i+1, col=1)
fig.write_image("../thesis/Images/discharging_experiment.pdf", "pdf")
fig.show()

In [67]:
for c_rate in C_RATES_CHARGE:
    power = 18.87 * c_rate * 256
    environment = vs.Environment(sim_start="2022-07-03 09:00:00")
    for battery in default_battery_models(initial_soc=0, np=16, ns=16, step_size=STEP_SIZE)[:4]:
        monitor = vs.Monitor(outfile=f"results/{type(battery).__name__}_Charge{c_rate}C.csv")
        environment.add_microgrid(
            actors=[vs.Actor(name=f"MockActor:{type(battery).__name__}", signal=vs.MockSignal(value=power))],
            storage=battery,
            controllers=[monitor],
            step_size=STEP_SIZE,
        )
    if c_rate < 0.7:
        environment.run(until=(1 / c_rate + 2 * c_rate) * 3600)
    else:
        environment.run(until=(1 / 0.7 + 0.7) * 3600)

2024-07-16 16:48:46.176 | INFO     | mosaik.scenario:start:311 - Starting "Actor" as "MockActor:SimpleBattery" ...
2024-07-16 16:48:46.177 | INFO     | mosaik.scenario:start:311 - Starting "Grid" as "Grid-0" ...
2024-07-16 16:48:46.178 | INFO     | mosaik.scenario:start:311 - Starting "Controller" as "Monitor-44" ...
2024-07-16 16:48:46.178 | INFO     | mosaik.scenario:start:311 - Starting "Storage" as "Storage-0" ...
2024-07-16 16:48:46.179 | INFO     | mosaik.scenario:start:311 - Starting "Actor" as "MockActor:CLCBattery" ...
2024-07-16 16:48:46.180 | INFO     | mosaik.scenario:start:311 - Starting "Grid" as "Grid-1" ...
2024-07-16 16:48:46.180 | INFO     | mosaik.scenario:start:311 - Starting "Controller" as "Monitor-45" ...
2024-07-16 16:48:46.181 | INFO     | mosaik.scenario:start:311 - Starting "Storage" as "Storage-1" ...
2024-07-16 16:48:46.181 | INFO     | mosaik.scenario:start:311 - Starting "Actor" as "MockActor:PybammBattery" ...
2024-07-16 16:48:46.182 | INFO     | mosaik.

In [28]:
fig = make_subplots(rows=3, cols=2, shared_yaxes=True, vertical_spacing=0.07, horizontal_spacing=0.05, subplot_titles=tuple(f"C-Rate {i}" for i in C_RATES_CHARGE))
fig.update_layout(
    width=800,
    height=800,
    margin={"r": 0, "l": 0, "t": 20, "b": 0},
    paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
    font_size=14,
    legend={"xanchor": "right", "x": 0.75},
)
fig.update_xaxes(
    {"showline": True, "linewidth": 1, "linecolor": "black", "ticks":"outside"}
)
fig.update_yaxes(
    {"showline": True, "linewidth": 1, "linecolor": "black", "ticks":"outside"}
)
showlegend = True
batteries = ["SimpleBattery", "CLCBattery", "PybammBattery", "LiionBatteryPack"]
colors = ["Blue", "Red", "Green", "Purple"]
for i, c_rate in enumerate(C_RATES_CHARGE):
    col = i % 2 + 1
    row = i // 2 + 1
    for battery, color in zip(batteries, colors[:4]):
        df = pd.read_csv(f"results/{battery}_Charge{c_rate}C.csv")
        fig.add_trace(
            go.Scatter(x=[i * STEP_SIZE / 3600 for i in range(len(df.index))], y=df["storage.soc"] * 100, showlegend=showlegend, line_color=color, name=battery), row=row, col=col
        )
    fig.update_xaxes(row=row, col=col, title="", ticksuffix="h")
    showlegend = False
for i in range(3):
    fig.update_yaxes(title="$\\text{State-of-Charge}$", ticksuffix = "%", row=i+1, col=1)
fig.write_image("../thesis/Images/charging_experiment.pdf", "pdf")
fig.show()

In [4]:
STEP_SIZE = 60

In [54]:
environment = vs.Environment(sim_start="2021-06-11 00:00:00")
for battery in default_battery_models(initial_soc=0.7, np=16, ns=16, step_size=STEP_SIZE)[:4]:
    environment.add_microgrid(
        actors=[
            vs.ComputingSystem(name=f"MockComputingSystem:{type(battery).__name__}", nodes=[vs.MockSignal(value=200), vs.MockSignal(value=50)], pue=1.3),
            vs.Actor(name=f"MockSolar:{type(battery).__name__}", signal=vs.HistoricalSignal(actual=pd.read_csv("data/solar_data.csv", index_col=0) * 10 * 0.15)),
        ],
        policy=MinSocPolicy(min_soc=0.6),
        storage=battery,
        controllers=[vs.Monitor(outfile=f"results/{type(battery).__name__}.csv"), ExperimentController()],
        step_size=STEP_SIZE,
    )

environment.run(until=48 * 3600)

2024-07-17 17:40:54.548 | INFO     | mosaik.scenario:start:311 - Starting "Actor" as "MockComputingSystem:SimpleBattery" ...
2024-07-17 17:40:54.549 | INFO     | mosaik.scenario:start:311 - Starting "Actor" as "MockSolar:SimpleBattery" ...
2024-07-17 17:40:54.549 | INFO     | mosaik.scenario:start:311 - Starting "Grid" as "Grid-0" ...
2024-07-17 17:40:54.550 | INFO     | mosaik.scenario:start:311 - Starting "Controller" as "Monitor-13" ...
2024-07-17 17:40:54.550 | INFO     | mosaik.scenario:start:311 - Starting "Controller" as "ExperimentController-0" ...
2024-07-17 17:40:54.551 | INFO     | mosaik.scenario:start:311 - Starting "Storage" as "Storage-0" ...
2024-07-17 17:40:54.554 | INFO     | mosaik.scenario:start:311 - Starting "Actor" as "MockComputingSystem:CLCBattery" ...
2024-07-17 17:40:54.554 | INFO     | mosaik.scenario:start:311 - Starting "Actor" as "MockSolar:CLCBattery" ...
2024-07-17 17:40:54.555 | INFO     | mosaik.scenario:start:311 - Starting "Grid" as "Grid-1" ...
202

In [78]:
battery = "SimpleBattery"
df = pd.read_csv(f"results/{battery}.csv", index_col=0)
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.05)
# First plot
fig.add_trace(
    go.Scatter(x=df.index, y=df[f"MockSolar:{battery}.p"], line_color="green", name="Solar"), row=1, col=1
)
line_dashes = ["dot", "dash"]
for i, col in enumerate(df.filter(regex="nodes").columns):
    fig.add_trace(
        go.Scatter(
            x=df.index, y=-df[col], line_color="red", line_dash=line_dashes[i], name=f"Node{i}",
        )
    )
fig.update_yaxes(title_text="$\\text{Power  } p^{<Actor>}$", row=1, col=1, ticksuffix="W")

# Second plot
pos_delta = df["p_delta"].apply(lambda x: max(x, 0))
neg_delta = df["p_delta"].apply(lambda x: min(x, 0))
fig.add_trace(
    go.Scatter(x=pos_delta.index, y=pos_delta, fill="tozeroy", line_color="green", showlegend=False), row=2, col=1
)
fig.add_trace(
    go.Scatter(x=neg_delta.index, y=neg_delta, fill="tozeroy", line_color="red", showlegend=False), row=2, col=1
)
fig.update_yaxes(title_text="$\\text{Power-Delta } \\Delta p$", row=2, col=1, ticksuffix=" W")

fig.update_layout(
    width=800,
    height=400,
    margin={"r": 0, "l": 0, "t": 0.1, "b": 0},
    legend={"x": 0.5, "y": 1, "yanchor": "top", "bgcolor": "rgba(0,0,0,0)"},
    paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
    font_size=14,
    xaxis_tickformat = '%d %B (%a)\n %Y'
)
fig.update_xaxes(showline=True, linewidth=1, linecolor="black", ticks="outside")
fig.update_yaxes(showline=True, linewidth=1, linecolor="black", ticks="outside")
fig.write_image("../thesis/Images/scenario_power.pdf", "pdf")
fig.show()

In [105]:
batteries = ["SimpleBattery", "CLCBattery", "PybammBattery", "LiionBatteryPack"]
colors = ["Blue", "Red", "Green", "Purple"]
fig = go.Figure()
for battery, color in zip(batteries, colors):
    df = pd.read_csv(f"results/{battery}.csv", index_col=0)
    fig.add_trace(
        go.Scatter(x=df.index, y=df["storage.soc"]*100, line_color=color, name=f"{battery}")
    )
fig.add_trace(
    go.Scatter(x=df.index, y=df["policy.min_soc"]*100, line_dash="dot", line_color="black", name="Min. SoC")
)
fig.update_yaxes(range=[20, 100], ticksuffix="%")

fig.update_layout(
    width=800,
    height=300,
    margin={"r": 0, "l": 0, "t": 20, "b": 0},
    paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
    legend={"x": 0.53, "y": 1, "yanchor": "top", "bgcolor": "rgba(0,0,0,0)"},
    font_size=14,
)
fig.update_xaxes(showline=True, linewidth=1, linecolor="black", ticks="outside")
fig.update_yaxes(showline=True, linewidth=1, linecolor="black", ticks="outside", title_text="$\\text{State-of-Charge}$")
fig.write_image("../thesis/Images/scenario_soc.pdf", "pdf")
fig.show()

In [34]:

batteries = ["SimpleBattery", "CLCBattery", "PybammBattery", "LiionBatteryPack"]
colors = ["Blue", "Red", "Green", "Purple"]
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.05)
for i, (battery, color) in enumerate(zip(["LiionBatteryPack", "PybammBattery", "CLCBattery", "SimpleBattery"], ["Purple", "Green", "Red", "Blue"])):
    df = pd.read_csv(f"results/{battery}.csv", index_col=0)
    e_delta = df["e_delta"]
    fill = "tozeroy" if i == 0 else "tonexty"
    fig.add_trace(
        go.Scatter(x=df.index, y=e_delta.clip(lower=0).cumsum() / 3600, line_color=color, fill=fill, name=battery), row=1, col=1
    )
for i, (battery, color) in enumerate(zip(["PybammBattery", "LiionBatteryPack", "CLCBattery", "SimpleBattery"], ["Green", "Purple", "Red", "Blue"])):
    df = pd.read_csv(f"results/{battery}.csv", index_col=0)
    e_delta = df["e_delta"]
    fill = "tozeroy" if i == 0 else "tonexty"
    fig.add_trace(
        go.Scatter(x=df.index, y=-e_delta.clip(upper=0).cumsum() / 3600, line_color=color, fill=fill, showlegend=False), row=2, col=1
    )
    print(e_delta.sum() / 3600)
    print(e_delta[e_delta > 0].sum() / 3600)
    print(e_delta[e_delta < 0].sum() / 3600)
    print(df["p_delta"].sum() * STEP_SIZE / 3600)
fig.update_layout(
    width=800,
    height=500,
    margin={"r": 0, "l": 0, "t": 20, "b": 0},
    paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
    legend={"x": 0.1, "y": 1, "yanchor": "top", "bgcolor": "rgba(0,0,0,0)"},
    font_size=14,
)
fig.update_xaxes(showline=True, linewidth=1, linecolor="black", ticks="outside")
fig.update_yaxes(showline=True, linewidth=1, linecolor="black", ticks="outside", ticksuffix="Wh")
fig.update_yaxes(row=1, col=1, title_text="$\\text{Cumulative Grid Charge}$")
fig.update_yaxes(row=2, col=1, title_text="$\\text{Cumulative Grid Discharge}$")
fig.write_image("../thesis/Images/scenario_grid_energy.pdf", "pdf")
fig.show()

-1081.6041463258014
2350.208813055667
-3431.8129593814688
-677.1
-1102.8599461257309
2332.3134395432535
-3435.1733856689843
-677.1
-1015.9023445135318
2463.5583531879024
-3479.4606977014337
-677.1
-778.9076666665476
2983.634000000119
-3762.5416666666665
-677.1


In [28]:
fig = go.Figure()
liion_e_delta = pd.read_csv("results/LiionBatteryPack.csv", index_col=0)["e_delta"]
for battery, color in zip(["SimpleBattery", "CLCBattery", "PybammBattery"], ["blue", "red", "green"]):
    df = pd.read_csv(f"results/{battery}.csv", index_col=0)
    e_delta = df["e_delta"].sub(liion_e_delta)
    fig.add_trace(
        go.Scatter(x=df.index, y=e_delta, line_color=color)
    )
fig.update_layout(
    width=800,
    height=500,
    margin={"r": 0, "l": 0, "t": 20, "b": 0},
    paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
    font_size=14,
)
fig.update_xaxes(showline=True, linewidth=1, linecolor="black", ticks="outside")
fig.update_yaxes(showline=True, linewidth=1, linecolor="black", ticks="outside", ticksuffix="Wh")
fig.show()

In [60]:
print(1102 / 1081)
print(1102 / 1016)
print(1102 / 779)

1.019426456984274
1.0846456692913387
1.4146341463414633


In [100]:
batteries = ["SimpleBattery", "CLCBattery", "PybammBattery", "LiionBatteryPack"]
colors = ["blue", "red", "green", "purple"]
fig = make_subplots(rows=1, cols=2, column_widths=[0.4, 0.6])
for battery, color in zip(batteries[:2], colors[:2]):
    df = pd.read_csv(f"results/Times_{battery}_16.csv")
    fig.add_trace(
        go.Box(y=df.iloc[:,0] * 1000, line_color=color, name=battery, showlegend=False), row=1, col=1
    )
for number_of_cells in [4, 8, 16, 32]:
    df = pd.read_csv(f"results/Times_LiionBatteryPack_{number_of_cells}.csv")
    fig.add_trace(
        go.Box(y=df.iloc[:,0], line_color="purple", name=f"LiionBatteryPack <br> {number_of_cells}S{number_of_cells}P", showlegend=False), row=1, col=2
    )
pybamm_median = np.median(np.loadtxt("results/Times_PybammBattery_16.csv"))
fig.add_shape(
    showlegend=True,
    type="line",
    line_color="green",
    xref="x2 domain",
    x0=0,
    x1=1,
    y0=pybamm_median,
    y1=pybamm_median,
    name="PybammBattery Median",
    row=1,
    col=2
)
fig.update_layout(
    width=800,
    height=500,
    margin={"r": 0, "l": 0, "t": 20, "b": 0},
    paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
    legend={"x": 0.5, "y": 1, "yanchor": "top", "bgcolor": "rgba(0,0,0,0)"},
    font_size=14,
)
fig.update_yaxes(showline=True, linewidth=1, linecolor="black", ticks="outside", ticksuffix="ms")
fig.update_xaxes(showline=True, linewidth=1, linecolor="black")
fig.update_yaxes(type="log", range=[-3, 0], row=1, col=2, ticksuffix="s")
fig.update_xaxes(row=1, col=2, tickfont_size=12)
fig.write_image("../thesis/Images/runtime_analysis.pdf", "pdf")
fig.show()

In [106]:
print(0.095/ 0.000191)
print(0.095/ 0.000196)
print(pybamm_median)
print(0.095 / pybamm_median)

497.38219895287955
484.6938775510204
0.007654044203333494
12.411739137673852
